# Assistiv Systems — FEP Real Data Fetcher
**Frailty Geography Intelligence · Layer 2 · Kent & Medway**

Fetches real NHS data from two sources and commits `kent-fep-data.json` to GitHub:
- **NHS Fingertips** — 10 outcomes indicators via `fingertips_py`
- **NHSBSA English Prescribing Dataset** — 5 prescribing signals via streaming CSV

### Before running:
1. Add your GitHub token to Colab Secrets (🔑 left sidebar)
   - Name: `GITHUB_TOKEN` · Value: your token with `public_repo` scope
2. `Runtime → Run all` — takes approximately 5–8 minutes

### To refresh quarterly:
- Update `EPD_URL` and `EPD_PERIOD` in Cell 1
- Latest EPD at: https://opendata.nhsbsa.net/dataset/english-prescribing-dataset-epd-with-snomed-code

---
*NHSBSA Copyright 2026 · OpenPrescribing.net, Bennett Institute, University of Oxford · OHID*

## Cell 1 — Configuration
Installs dependencies and sets all constants. Update `EPD_URL` and `EPD_PERIOD` each quarter.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "fingertips_py", "requests", "pandas", "--quiet"], check=True)
print("Dependencies installed")

import requests, pandas as pd, fingertips_py as ftp
import json, csv, base64
from datetime import datetime, timezone
from collections import defaultdict
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-fep-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
print(f"GitHub token: {'Found' if GITHUB_TOKEN else 'MISSING - add to Colab Secrets'}")

KENT_ICB_ONS   = "E54000032"
KENT_ICB_CODE  = "QKS"
KENT_COUNTY    = "E10000016"
KENT_LIST_SIZE = 1_900_000
ENGLAND        = "E92000001"

# UPDATE THESE EACH QUARTER
EPD_URL    = "https://opendata.nhsbsa.net/dataset/906115a6-4155-44be-8b81-f8e83cebfb84/resource/6b52baa3-c550-4b1d-8cd2-46d40d840353/download/epd_snomed_202603.csv"
EPD_PERIOD = "Mar 2026"

ENGLAND_PRESCRIBING_RATES = {
    'hypnotics': 10.2, 'antidepressants': 110.0,
    'bisphosphonates': 6.8, 'diuretics': 2.5, 'ace_arb': 95.0,
}

print(f"Config loaded | ICB: {KENT_ICB_ONS} | EPD: {EPD_PERIOD}")


## Cell 2 — Fetch NHS Fingertips Indicators
10 real outcomes indicators for Kent & Medway ICB via `fingertips_py`.
Fetches falls admissions, hip fractures, winter mortality, loneliness, dementia, social isolation, fuel poverty.

In [ ]:
FINGERTIPS_INDICATORS = {
    'falls_65':           (22401, 'Falls admissions 65+',        KENT_ICB_ONS),
    'falls_65_79':        (22402, 'Falls admissions 65-79',      KENT_ICB_ONS),
    'falls_80':           (22403, 'Falls admissions 80+',        KENT_ICB_ONS),
    'winter_mortality':   (90360, 'Winter mortality index',      KENT_COUNTY),
    'loneliness':         (94175, 'Loneliness often/always',     KENT_ICB_ONS),
    'social_isolation':   (90280, 'Social isolation - SC users', KENT_COUNTY),
    'dementia_diagnosis': (92949, 'Dementia diagnosis rate 65+', KENT_ICB_ONS),
    'hip_fractures_65':   (41401, 'Hip fractures 65+',           KENT_ICB_ONS),
    'hip_fractures_80':   (41403, 'Hip fractures 80+',           KENT_ICB_ONS),
    'fuel_poverty':       (93759, 'Fuel poverty',                KENT_ICB_ONS),
}

fingertips_results = {}
print("Fetching NHS Fingertips indicators...")
print(f"{'Indicator':<35} {'Kent':>8}  {'England':>8}  Period")
print("-" * 70)

for key, (ind_id, label, area_code) in FINGERTIPS_INDICATORS.items():
    try:
        data = ftp.get_data_for_indicator_at_all_available_geographies(ind_id)
        if data is None:
            raise ValueError("Returned None")
        kent = data[data['Area Code'] == area_code].sort_values('Time period')
        eng  = data[data['Area Code'] == ENGLAND].sort_values('Time period')
        if len(kent) == 0:
            raise ValueError(f"No data for {area_code}")
        kent_val = round(float(kent.tail(1)['Value'].values[0]), 2)
        eng_val  = round(float(eng.tail(1)['Value'].values[0]), 2) if len(eng) else None
        period   = str(kent.tail(1)['Time period'].values[0])
        fingertips_results[key] = {
            'value': kent_val, 'england': eng_val,
            'period': period, 'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }
        direction = "up" if eng_val and kent_val > eng_val else "down"
        print(f"  {label:<33} {kent_val:>8}  {str(eng_val):>8}  {period}  {direction}")
    except Exception as e:
        print(f"  FAILED {label:<29} -- {e}")
        fingertips_results[key] = {
            'value': None, 'england': None, 'period': None,
            'source': f'NHS Fingertips indicator {ind_id}', 'label': label,
        }

real_ft = sum(1 for v in fingertips_results.values() if v['value'])
print(f"\n{real_ft}/{len(FINGERTIPS_INDICATORS)} Fingertips indicators fetched")


## Cell 3 — Stream NHSBSA Prescribing Data
Streams the EPD CSV line by line (~18M rows). Filters for Kent & Medway and 5 BNF signal groups.
**Takes 3–5 minutes** — progress shown every 2M rows.

In [ ]:
ICB_I=3; BNF_I=14; CHEM_I=15; ITEMS_I=20; QTY_I=21
KENT_FILTER = 'KENT AND MEDWAY'
TARGET_BNF  = ('0401010','0403','060602','0201','0205')

signal_items = defaultdict(int)
signal_quantity = defaultdict(float)
signal_drugs = defaultdict(set)
rows_read = kent_rows = 0
buffer = ""; header_done = False

print(f"Streaming NHSBSA EPD -- {EPD_PERIOD}")
print("Progress every 2M rows. Takes 3-5 minutes.\n")

try:
    with requests.get(EPD_URL, stream=True, timeout=300,
                      headers={"User-Agent": "Mozilla/5.0 AssistivSystems/1.0"}) as resp:
        print(f"HTTP {resp.status_code}")
        if resp.status_code != 200:
            raise ValueError(f"HTTP {resp.status_code}")
        for raw_chunk in resp.iter_content(chunk_size=512*1024):
            buffer += raw_chunk.decode('utf-8', errors='replace')
            lines = buffer.split('\n'); buffer = lines[-1]
            for line in lines[:-1]:
                line = line.strip()
                if not line: continue
                if not header_done:
                    header_done = True; continue
                rows_read += 1
                if rows_read % 2_000_000 == 0:
                    print(f"  {rows_read:>12,} rows | {kent_rows:>6,} Kent | {dict(signal_items)}")
                if KENT_FILTER not in line: continue
                try:
                    row = next(csv.reader([line]))
                except Exception:
                    continue
                if len(row) <= max(ICB_I, BNF_I, ITEMS_I, QTY_I): continue
                icb = row[ICB_I]; bnf = row[BNF_I].strip('"')
                if KENT_FILTER not in icb: continue
                if not bnf.startswith(TARGET_BNF): continue
                kent_rows += 1
                try:
                    items = int(row[ITEMS_I]); qty = float(row[QTY_I]); chem = row[CHEM_I]
                    if   bnf.startswith('0401010'): key = 'hypnotics'
                    elif bnf.startswith('0403'):    key = 'antidepressants'
                    elif bnf.startswith('060602'):  key = 'bisphosphonates'
                    elif bnf.startswith('0201'):    key = 'diuretics'
                    elif bnf.startswith('0205'):    key = 'ace_arb'
                    else:                           key = 'other'
                    signal_items[key] += items
                    signal_quantity[key] += qty
                    signal_drugs[key].add(chem)
                except (ValueError, IndexError):
                    continue
except Exception as e:
    import traceback; print(f"\nError at row {rows_read:,}: {e}"); traceback.print_exc()

print(f"\nCOMPLETE -- {rows_read:,} rows, {kent_rows:,} Kent matched")
for key in ['hypnotics','antidepressants','bisphosphonates','diuretics','ace_arb']:
    print(f"  {key:<20} {signal_items.get(key,0):>10,} items")


## Cell 4 — Calculate Prescribing Rates
Converts raw item counts to rates per 1,000 patients and compares to England averages.

In [ ]:
epd_results = {}
print(f"Kent & Medway prescribing rates vs England ({EPD_PERIOD})")
print(f"List size: {KENT_LIST_SIZE:,}")
print(f"\n  {'Signal':<20} {'Kent/1k':>9}  {'Eng/1k':>9}  {'Ratio':>7}  Direction")
print(f"  {'-'*60}")

for key in ['hypnotics','antidepressants','bisphosphonates','diuretics','ace_arb']:
    items  = signal_items.get(key, 0)
    qty    = signal_quantity.get(key, 0.0)
    drugs  = len(signal_drugs.get(key, set()))
    kent_r = round((items / KENT_LIST_SIZE) * 1000, 3) if items else 0
    eng_r  = ENGLAND_PRESCRIBING_RATES.get(key, 0)
    ratio  = round(kent_r / eng_r, 3) if eng_r and kent_r else 0
    direction = "above" if ratio > 1.05 else "below" if ratio < 0.95 else "similar"
    epd_results[key] = {
        'rate_per_1000': kent_r, 'england': eng_r, 'items': items,
        'qty': round(qty), 'substances': drugs, 'ratio': ratio,
        'period': EPD_PERIOD,
        'source': f'NHSBSA EPD EPD_SNOMED_{EPD_PERIOD.replace(" ","").upper()}',
    }
    print(f"  {key:<20} {kent_r:>9.3f}  {eng_r:>9.1f}  {ratio:>7.3f}  {direction}")

real_epd = sum(1 for v in epd_results.values() if v['rate_per_1000'] > 0)
print(f"\n{real_epd}/{len(epd_results)} EPD signals computed")
total_real = sum(1 for v in fingertips_results.values() if v['value']) + real_epd
print(f"Total real signals: {total_real}/15")


## Cell 5 — Build District FEP Scores & Commit to GitHub
Normalises all signals to 0–100, applies district demographic profiles,
computes weighted FEP scores for 13 districts, and commits `kent-fep-data.json` to GitHub.

In [ ]:
SIGNAL_NAMES = [
    "Over-75s Living Alone",
    "Falls Admissions 65+",
    "Hip Fracture Rate 65+",
    "Deprivation (IMD)",
    "Winter Mortality Index",
    "Care Home Gap",
    "Loneliness Rate",
    "Dementia Diagnosis Rate",
    "Hip Fractures 80+",
    "Social Isolation Rate",
    "Hypnotics Prescribing",
    "Antidepressant Rate",
    "Bisphosphonate Rate",
    "Diuretics Rate",
    "ACE/ARB Prescribing",
]

WEIGHTS = [0.14,0.13,0.10,0.09,0.08,0.07,0.07,0.06,0.05,0.05,0.05,0.05,0.04,0.01,0.01]
assert abs(sum(WEIGHTS) - 1.0) < 0.001, f"Weights sum to {sum(WEIGHTS)}"

def norm(value, england, invert=False):
    if not value or not england: return 50.0
    score = (value / england) * 50
    return round(min(100, max(0, 100 - score if invert else score)), 1)

def ft(key, invert=False):
    v = fingertips_results.get(key, {})
    return norm(v.get('value'), v.get('england'), invert)

def epd(key):
    v = epd_results.get(key, {})
    return norm(v.get('rate_per_1000'), ENGLAND_PRESCRIBING_RATES.get(key))

ICB_BASE = [
    50.0, ft('falls_65'), ft('hip_fractures_65'), 50.0,
    ft('winter_mortality'), 50.0, ft('loneliness'),
    ft('dementia_diagnosis', invert=True), ft('hip_fractures_80'),
    ft('social_isolation'), epd('hypnotics'), epd('antidepressants'),
    epd('bisphosphonates'), epd('diuretics'), epd('ace_arb'),
]

print("ICB normalised scores (England avg = 50):")
for name, score in zip(SIGNAL_NAMES, ICB_BASE):
    tag = "REAL" if score != 50.0 else "synth"
    bar = ">" if score > 50 else "<" if score < 50 else "="
    print(f"  {bar} {score:>5.1f}  [{tag:>5}]  {name}")

PROFILES = {
    "Thanet":              [1.30,1.25,1.20,1.35,1.28,1.30,1.25,1.18,1.20,1.22,1.28,1.25,1.22,1.20,1.18],
    "Folkestone & Hythe":  [1.22,1.18,1.15,1.22,1.20,1.20,1.18,1.12,1.15,1.15,1.18,1.15,1.12,1.10,1.10],
    "Dover":               [1.18,1.15,1.12,1.18,1.15,1.10,1.14,1.08,1.10,1.10,1.15,1.12,1.10,1.08,1.08],
    "Swale":               [1.12,1.10,1.08,1.12,1.10,1.12,1.08,1.05,1.08,1.05,1.10,1.08,1.08,1.05,1.05],
    "Medway":              [1.06,1.05,1.04,1.08,1.05,1.08,1.02,1.02,1.05,1.02,1.05,1.05,1.04,1.02,1.02],
    "Gravesham":           [1.00,0.98,1.02,1.02,1.00,1.05,0.98,1.00,1.02,1.00,1.00,0.98,1.00,1.00,1.00],
    "Ashford":             [0.96,0.95,0.98,0.98,0.96,1.00,0.94,0.96,0.98,0.95,0.95,0.95,0.96,0.95,0.95],
    "Canterbury":          [0.90,0.90,0.92,0.85,0.92,0.92,0.92,0.90,0.90,0.90,0.90,0.90,0.90,0.88,0.88],
    "Dartford":            [0.88,0.88,0.90,0.95,0.88,0.90,0.88,0.88,0.88,0.88,0.88,0.88,0.88,0.88,0.88],
    "Maidstone":           [0.85,0.85,0.88,0.95,0.85,0.92,0.85,0.85,0.85,0.85,0.85,0.85,0.85,0.85,0.85],
    "Tonbridge & Malling": [0.78,0.78,0.82,0.78,0.80,0.85,0.80,0.80,0.80,0.80,0.78,0.78,0.80,0.80,0.80],
    "Sevenoaks":           [0.65,0.65,0.68,0.52,0.65,0.75,0.65,0.65,0.65,0.65,0.62,0.62,0.65,0.65,0.65],
    "Tunbridge Wells":     [0.58,0.58,0.62,0.58,0.60,0.65,0.60,0.60,0.60,0.60,0.55,0.55,0.60,0.60,0.60],
}

LAD_CODES = {
    "Thanet":"E07000114","Folkestone & Hythe":"E07000108","Dover":"E07000105",
    "Swale":"E07000113","Medway":"E06000035","Gravesham":"E07000109",
    "Ashford":"E07000106","Canterbury":"E07000107","Dartford":"E07000108",
    "Maidstone":"E07000110","Tonbridge & Malling":"E07000115",
    "Sevenoaks":"E07000111","Tunbridge Wells":"E07000116",
}

POP75 = {
    "Thanet":18200,"Folkestone & Hythe":14100,"Dover":13800,"Swale":15200,
    "Medway":19400,"Gravesham":11800,"Ashford":13600,"Canterbury":16300,
    "Dartford":10800,"Maidstone":16700,"Tonbridge & Malling":13100,
    "Sevenoaks":12100,"Tunbridge Wells":11200,
}

districts = []
for name, profile in PROFILES.items():
    signals = [round(min(100, max(0, ICB_BASE[i] * profile[i])), 1) for i in range(15)]
    fep  = round(min(100, max(0, sum(s * w for s, w in zip(signals, WEIGHTS)))))
    risk = "critical" if fep >= 70 else "high" if fep >= 55 else "moderate" if fep >= 40 else "low"
    districts.append({"name": name, "lad_code": LAD_CODES[name], "fep": fep,
                       "risk": risk, "signals": signals, "signal_names": SIGNAL_NAMES,
                       "pop75": POP75[name]})

districts.sort(key=lambda x: x["fep"], reverse=True)

print("\nFINAL DISTRICT FEP SCORES:")
print(f"  {'District':<25} {'FEP':>5}  Risk")
print("  " + "-" * 40)
for d in districts:
    marker = " <--" if d['risk'] == 'critical' else ""
    print(f"  {d['name']:<25} {d['fep']:>5}  {d['risk']}{marker}")

real_signals = (
    [k for k, v in fingertips_results.items() if v.get('value')] +
    [k for k, v in epd_results.items() if v.get('rate_per_1000', 0) > 0]
)

output = {
    "meta": {
        "generated":         datetime.now(timezone.utc).isoformat(),
        "description":       "Kent & Medway FEP scores -- Assistiv Systems Layer 2",
        "icb":               "NHS Kent and Medway ICB (QKS)",
        "icb_ons_code":      KENT_ICB_ONS,
        "data_quality":      f"real -- {len(real_signals)} real signals",
        "signals_real":      real_signals,
        "signals_synthetic": ["alone_75", "deprivation_imd", "care_home_gap"],
        "signal_names":      SIGNAL_NAMES,
        "weights":           WEIGHTS,
        "sources": {
            "fingertips":  "NHS Fingertips/OHID PHOF via fingertips_py",
            "epd":         f"NHSBSA English Prescribing Dataset -- {EPD_PERIOD}",
            "demographic": "ONS Census 2021 | IMD 2019 | CQC register",
        },
        "note": "District scores = ICB real values x demographic profiles. Sub-ICB linkage Phase 2.",
    },
    "icb_baseline": {
        "fingertips":  fingertips_results,
        "prescribing": epd_results,
    },
    "districts": districts,
}

def commit_to_github(content, repo, filepath, token):
    api_url = f"https://api.github.com/repos/{repo}/contents/{filepath}"
    headers = {"Authorization": "token " + token, "Accept": "application/vnd.github.v3+json"}
    b64 = base64.b64encode(json.dumps(content, indent=2).encode()).decode()
    r = requests.get(api_url, headers=headers)
    sha = r.json().get("sha") if r.status_code == 200 else None
    payload = {
        "message": f"FEP refresh -- {datetime.now(timezone.utc).strftime('%Y-%m-%d')} -- {len(real_signals)} real signals",
        "content": b64, "branch": "main",
    }
    if sha: payload["sha"] = sha
    r = requests.put(api_url, headers=headers, json=payload)
    if r.status_code in (200, 201):
        print(f"\nCommitted to GitHub")
        print(f"  File:   https://raw.githubusercontent.com/{repo}/main/{filepath}")
        print(f"  Commit: {r.json().get('commit',{}).get('html_url','')}")
        return True
    print(f"\nCommit failed: {r.status_code} -- {r.json().get('message','')}")
    return False

print("\nCommitting to GitHub...")
commit_to_github(output, GITHUB_REPO, GITHUB_FILE, GITHUB_TOKEN)
print("\nDone. Run quarterly to refresh.")
